In [ ]:
# Cell 1 — Imports and config (multi-PDB only)
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from Bio.PDB import PDBParser, PDBIO, Select

try:
    import pdbfixer
    from openmm.app import PDBFile
except ImportError as e:
    raise ImportError("Install pdbfixer and openmm: conda install -c conda-forge pdbfixer openmm") from e

# --- Configurable parameters ---
INPUT_PDB_DIR = "all_pdbs"  # folder containing .pdb files (protein + UNL ligand)
OUT_DIR_PREFIX = "fgo_out"  # per-PDB output: {OUT_DIR_PREFIX}_{pdb_stem}
TARGET_RESIDUES = ["GLN853", "MET865", "LYS883", "TYR934", "GLY1132"]
CHAIN = "A"
CLOSE_CUTOFF = 4.5  # angstroms
MAX_MODS_PER_SITE = 3
SEED = 42
FIXER_PH = 7.0
MAX_VARIANTS_PER_PARENT = 12
LIGAND_RESNAME = "UNL"  # only process this heterogen; PTR etc. ignored

INPUT_PDB_DIR = Path(INPUT_PDB_DIR)
if not INPUT_PDB_DIR.is_dir():
    raise ValueError(f"INPUT_PDB_DIR must be a directory: {INPUT_PDB_DIR}")
run_list = [(str(p.resolve()), f"{OUT_DIR_PREFIX}_{p.stem}") for p in sorted(INPUT_PDB_DIR.glob("*.pdb"))]
if not run_list:
    raise ValueError(f"No .pdb files in {INPUT_PDB_DIR}")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)
np.random.seed(SEED)
logger.info(f"run_list: {len(run_list)} PDB(s)")

In [ ]:
# Cell 2 — Edit utilities (RDKit RWMol)
def is_heavy_atom(mol, idx):
    return mol is not None and 0 <= idx < mol.GetNumAtoms() and mol.GetAtomWithIdx(idx).GetSymbol() != "H"

def safe_sanitize(mol):
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

def add_methyl(parent_mol, atom_idx):
    rw = Chem.RWMol(parent_mol)
    if not is_heavy_atom(rw, atom_idx):
        return None, None
    new_c = rw.AddAtom(Chem.Atom(6))
    rw.AddBond(atom_idx, new_c, Chem.BondType.SINGLE)
    for _ in range(3):
        h = rw.AddAtom(Chem.Atom(1))
        rw.AddBond(new_c, h, Chem.BondType.SINGLE)
    mol = safe_sanitize(rw)
    if mol is None:
        return None, None
    return Chem.MolToSmiles(mol), mol

def add_methylene(parent_mol, atom_idx, n=1):
    rw = Chem.RWMol(parent_mol)
    if not is_heavy_atom(rw, atom_idx):
        return None, None
    prev = atom_idx
    for _ in range(n):
        new_c = rw.AddAtom(Chem.Atom(6))
        rw.AddBond(prev, new_c, Chem.BondType.SINGLE)
        for _ in range(2):
            h = rw.AddAtom(Chem.Atom(1))
            rw.AddBond(new_c, h, Chem.BondType.SINGLE)
        prev = new_c
    mol = safe_sanitize(rw)
    if mol is None:
        return None, None
    return Chem.MolToSmiles(mol), mol

def replace_halogen(parent_mol, atom_idx, target_halogen):
    rw = Chem.RWMol(parent_mol)
    if not is_heavy_atom(rw, atom_idx):
        return None, None
    sym = rw.GetAtomWithIdx(atom_idx).GetSymbol()
    if sym not in ("F", "Cl", "Br", "I"):
        return None, None
    rw.GetAtomWithIdx(atom_idx).SetAtomicNum({"F": 9, "Cl": 17, "Br": 35, "I": 53}.get(target_halogen, 17))
    mol = safe_sanitize(rw)
    if mol is None:
        return None, None
    return Chem.MolToSmiles(mol), mol

def attach_substituent(parent_mol, atom_idx, sub_smiles):
    sub = Chem.MolFromSmiles(sub_smiles)
    if sub is None:
        return None, None
    rw = Chem.RWMol(parent_mol)
    if not is_heavy_atom(rw, atom_idx):
        return None, None
    merge = Chem.CombineMols(rw, sub)
    rw = Chem.RWMol(merge)
    n_parent = parent_mol.GetNumAtoms()
    rw.AddBond(atom_idx, n_parent, Chem.BondType.SINGLE)
    mol = safe_sanitize(rw)
    if mol is None:
        return None, None
    return Chem.MolToSmiles(mol), mol

def add_OH_or_NH2(parent_mol, atom_idx, type_="OH"):
    rw = Chem.RWMol(parent_mol)
    if not is_heavy_atom(rw, atom_idx):
        return None, None
    if type_.upper() == "OH":
        o = rw.AddAtom(Chem.Atom(8))
        rw.AddBond(atom_idx, o, Chem.BondType.SINGLE)
        h = rw.AddAtom(Chem.Atom(1))
        rw.AddBond(o, h, Chem.BondType.SINGLE)
    else:
        n = rw.AddAtom(Chem.Atom(7))
        rw.AddBond(atom_idx, n, Chem.BondType.SINGLE)
        for _ in range(2):
            h = rw.AddAtom(Chem.Atom(1))
            rw.AddBond(n, h, Chem.BondType.SINGLE)
    mol = safe_sanitize(rw)
    if mol is None:
        return None, None
    return Chem.MolToSmiles(mol), mol

logger.info("Edit utilities defined.")

In [ ]:
# Cell 3 — Residue-specific modification rules (JAK2: GLN, MET, LYS, TYR)
def get_residue_name_from_label(label):
    s = (label or "").strip().upper()
    num = ""
    for c in reversed(s):
        if c.isdigit():
            num = c + num
        else:
            break
    return s[: len(s) - len(num)].strip() or None

def _atom_is_carbon(mol, idx):
    return mol.GetAtomWithIdx(idx).GetSymbol() == "C"

def _atom_is_quaternary(mol, idx):
    return mol.GetAtomWithIdx(idx).GetDegree() >= 4

def _atom_eligible_GLN(mol, idx):
    return is_heavy_atom(mol, idx) and _atom_is_carbon(mol, idx) and not _atom_is_quaternary(mol, idx)

def _atom_eligible_MET(mol, idx):
    return is_heavy_atom(mol, idx) and _atom_is_carbon(mol, idx)

def _atom_eligible_LYS(mol, idx):
    return is_heavy_atom(mol, idx)

def _atom_eligible_TYR(mol, idx):
    return is_heavy_atom(mol, idx) and mol.GetAtomWithIdx(idx).GetSymbol() == "C"

def generate_residue_specific_modifications(parent_mol, atom_idx, residue_name, max_mods=MAX_MODS_PER_SITE):
    out = []
    res = (residue_name or "").upper()
    if res == "GLN":
        if not _atom_eligible_GLN(parent_mol, atom_idx):
            return out
        for sub_smi, rule_cat, mod_type in [("C(=O)(N)N", "GLN_amide", "attach_CONH2"), ("C(=O)C", "GLN_hbond_acceptor", "attach_acetyl")]:
            if len(out) >= max_mods:
                break
            smi, mol = attach_substituent(parent_mol, atom_idx, sub_smi)
            if mol and smi:
                out.append((smi, mol, rule_cat, mod_type))
        for typ, rule_cat, mod_type in [("NH2", "GLN_hbond_donor", "attach_NH2"), ("OH", "GLN_hydroxyl", "attach_OH")]:
            if len(out) >= max_mods:
                break
            smi, mol = add_OH_or_NH2(parent_mol, atom_idx, typ)
            if mol and smi:
                out.append((smi, mol, rule_cat, mod_type))
    elif res == "MET":
        if not _atom_eligible_MET(parent_mol, atom_idx):
            return out
        smi, mol = add_methyl(parent_mol, atom_idx)
        if mol and smi:
            out.append((smi, mol, "MET_hydrophobic", "add_methyl"))
        if len(out) < max_mods:
            smi, mol = add_methylene(parent_mol, atom_idx, n=1)
            if mol and smi:
                out.append((smi, mol, "MET_hydrophobic", "add_methylene_n1"))
        if len(out) < max_mods:
            smi, mol = attach_substituent(parent_mol, atom_idx, "CC")
            if mol and smi:
                out.append((smi, mol, "MET_hydrophobic", "attach_ethyl"))
        if len(out) < max_mods:
            smi, mol = attach_substituent(parent_mol, atom_idx, "C(C)C")
            if mol and smi:
                out.append((smi, mol, "MET_hydrophobic", "attach_isopropyl"))
    elif res == "LYS":
        if not _atom_eligible_LYS(parent_mol, atom_idx):
            return out
        for sub_smi, rule_cat, mod_type in [("C(=O)C", "LYS_hbond_acceptor", "attach_acetyl"), ("C#N", "LYS_nitrile", "attach_nitrile"), ("c1ccccc1", "LYS_aromatic", "attach_phenyl")]:
            if len(out) >= max_mods:
                break
            smi, mol = attach_substituent(parent_mol, atom_idx, sub_smi)
            if mol and smi:
                out.append((smi, mol, rule_cat, mod_type))
    elif res == "TYR":
        if not _atom_eligible_TYR(parent_mol, atom_idx):
            return out
        for sub_smi, rule_cat, mod_type in [("c1ccccc1", "TYR_aromatic", "attach_phenyl"), ("c1ccncc1", "TYR_heterocycle", "attach_pyridine")]:
            if len(out) >= max_mods:
                break
            smi, mol = attach_substituent(parent_mol, atom_idx, sub_smi)
            if mol and smi:
                out.append((smi, mol, rule_cat, mod_type))
        if len(out) < max_mods:
            smi, mol = add_OH_or_NH2(parent_mol, atom_idx, "OH")
            if mol and smi:
                out.append((smi, mol, "TYR_hydroxyl", "attach_OH"))
    else:
        if not is_heavy_atom(parent_mol, atom_idx):
            return out
        for fn, args, rule_cat, mod_type in [(add_methyl, (parent_mol, atom_idx), "generic", "add_methyl"), (lambda m, i: add_methylene(m, i, 1), (parent_mol, atom_idx), "generic", "add_methylene_n1"), (lambda m, i: add_OH_or_NH2(m, i, "OH"), (parent_mol, atom_idx), "generic", "attach_OH")]:
            if len(out) >= max_mods:
                break
            smi, mol = fn(*args)
            if mol and smi:
                out.append((smi, mol, rule_cat, mod_type))
    return out[:max_mods]

logger.info("Residue-specific rules defined (GLN, MET, LYS, TYR).")

In [ ]:
# Cell 4 — Multi-PDB runner: extract, fix, map, modify, save per PDB + master CSV
from collections import defaultdict
from rdkit.Chem.inchi import MolToInchiKey

def parse_target_label(label):
    s = label.strip().upper()
    num = ""
    for c in reversed(s):
        if c.isdigit():
            num = c + num
        else:
            break
    resname = s[: len(s) - len(num)].strip() or None
    resnum = int(num) if num else None
    return resname, resnum

def _json_safe(obj):
    """Convert PDBFixer report to JSON-serializable (keys/values may be Residue etc)."""
    if obj is None or isinstance(obj, (bool, int, float, str)):
        return obj
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(x) for x in obj]
    return str(obj)

# #region agent log (writes to cwd so notebook runs in Jupyter Lab / any env)
_dbg_path = str(Path.cwd() / "debug_fgo_97158e.log")
def _dbg(msg, data, hid):
    try:
        import time as _t
        with open(_dbg_path, "a") as _f:
            _f.write(json.dumps({"sessionId": "97158e", "message": msg, "data": data, "timestamp": int(_t.time()*1000), "hypothesisId": hid}) + chr(10))
    except Exception:
        pass
# #endregion

all_tables = []
for input_pdb, out_dir in run_list:
    pdb_stem = Path(input_pdb).stem
    _dbg("loop_start", {"input_pdb": input_pdb, "out_dir": out_dir}, "H0")
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for h in list(logger.handlers):
        if getattr(h, "baseFilename", None):
            logger.removeHandler(h)
    logger.addHandler(logging.FileHandler(Path(out_dir) / "log.txt"))
    logger.info(f"Processing {input_pdb} -> {out_dir}")
    parents_meta = []
    mapping_records = []
    variants_rows = []
    seen_inchikey = set()
    duplicates_skipped = 0
    candidate_sites = 0
    var_count_per_parent = defaultdict(int)

    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("combined", input_pdb)
    def is_het(residue):
        return residue.id[0] != " "
    seen = set()
    for model in struct:
        for chain in model:
            for res in chain:
                if not is_het(res) or res.get_resname().strip() != LIGAND_RESNAME:
                    continue
                key = (res.get_resname().strip(), res.id[1], chain.id)
                if key in seen:
                    continue
                seen.add(key)
                parent_id = f"{pdb_stem}_{res.get_resname().strip()}_{chain.id}_{res.id[1]}"
                lig_pdb_path = Path(out_dir) / f"ligand_{parent_id}.pdb"
                io = PDBIO()
                class LigSelect(Select):
                    def accept_residue(self, residue):
                        return residue.id == res.id and residue.get_parent().id == chain.id
                io.set_structure(struct)
                io.save(str(lig_pdb_path), LigSelect())
                mol = Chem.MolFromPDBFile(str(lig_pdb_path), removeHs=False, sanitize=False)
                if mol is None:
                    mol = Chem.MolFromPDBFile(str(lig_pdb_path), removeHs=False, sanitize=True)
                if mol is None:
                    logger.warning(f"Skip {parent_id}: RDKit could not parse")
                    continue
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
                lig_sdf_path = Path(out_dir) / f"ligand_{parent_id}.sdf"
                w = Chem.SDWriter(str(lig_sdf_path))
                w.write(mol)
                w.close()
                parents_meta.append({"parent_id": parent_id, "resname": res.get_resname().strip(), "resnum": res.id[1], "chain": chain.id, "lig_pdb_path": str(lig_pdb_path), "lig_sdf_path": str(lig_sdf_path)})
    _dbg("after_extract", {"n_parents": len(parents_meta)}, "H1")
    if not parents_meta:
        logger.warning(f"No {LIGAND_RESNAME} in {input_pdb}; skipping")
        continue
    with open(Path(out_dir) / "parents_meta.json", "w") as f:
        json.dump({e["parent_id"]: e for e in parents_meta}, f, indent=2)

    class ProteinOnlySelect(Select):
        def accept_residue(self, residue):
            return residue.id[0] == " "
    protein_only_path = Path(out_dir) / "protein_only.pdb"
    io = PDBIO()
    io.set_structure(struct)
    io.save(str(protein_only_path), ProteinOnlySelect())
    fixer = pdbfixer.PDBFixer(filename=str(protein_only_path))
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=FIXER_PH)
    protein_fixed_path = Path(out_dir) / "protein_fixed.pdb"
    with open(protein_fixed_path, "w") as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    fix_report = {"missing_residues": getattr(fixer, "missingResidues", {}), "missing_atoms": getattr(fixer, "missingAtoms", {}), "missing_terminals": getattr(fixer, "missingTerminals", {})}
    with open(Path(out_dir) / "protein_fix_report.json", "w") as f:
        json.dump(_json_safe(fix_report), f, indent=2)

    BACKBONE_NAMES = {"N", "CA", "C", "O"}
    struct_fixed = parser.get_structure("fixed", str(protein_fixed_path))
    centroids = {}
    for label in TARGET_RESIDUES:
        resname, resnum = parse_target_label(label)
        if resnum is None:
            continue
        coords = []
        for model in struct_fixed:
            for chain in model:
                res_id = (" ", resnum, " ")
                if res_id not in chain:
                    continue
                res = chain[res_id]
                if resname and res.get_resname().strip() != resname:
                    continue
                for atom in res:
                    if atom.name in BACKBONE_NAMES or atom.name.startswith("H"):
                        continue
                    coords.append(atom.get_coord())
                break
            if coords:
                break
        if coords:
            centroids[label] = np.array(coords).mean(axis=0).tolist()
    with open(Path(out_dir) / "residue_centroids.json", "w") as f:
        json.dump(centroids, f, indent=2)

    for entry in parents_meta:
        mol = Chem.MolFromMolFile(entry["lig_sdf_path"], removeHs=False)
        if mol is None:
            mol = Chem.MolFromPDBFile(entry["lig_pdb_path"], removeHs=False)
        if mol is None:
            continue
        conf = mol.GetConformer()
        if conf is None:
            continue
        for target_label, centroid in centroids.items():
            centroid_arr = np.array(centroid)
            dists = [(i, np.linalg.norm(np.array([conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]) - centroid_arr)) for i in range(mol.GetNumAtoms()) if mol.GetAtomWithIdx(i).GetSymbol() != "H"]
            if not dists:
                continue
            dists.sort(key=lambda x: x[1])
            nearest_atom_idx = dists[0][0]
            close_atom_idxs = [i for i, d in dists if d <= CLOSE_CUTOFF]
            distances = {i: d for i, d in dists}
            mapping_records.append({"parent_id": entry["parent_id"], "resname": entry["resname"], "resnum": entry["resnum"], "chain": entry["chain"], "target_residue": target_label, "nearest_atom_idx": nearest_atom_idx, "close_atom_idxs": close_atom_idxs, "distances": {str(k): round(v, 3) for k, v in distances.items()}})
    with open(Path(out_dir) / "mapping.json", "w") as f:
        json.dump(mapping_records, f, indent=2)
    _n_close = sum(len(r.get("close_atom_idxs", [])) for r in mapping_records)
    _dbg("after_mapping", {"n_mapping": len(mapping_records), "n_close_atoms_total": _n_close}, "H3")

    debug_path = Path(out_dir) / "debug.jsonl"
    with open(debug_path, "w") as f:
        pass
    def log_debug(obj):
        with open(debug_path, "a") as f:
            f.write(json.dumps(obj) + chr(10))
    parent_mols = {}
    for entry in parents_meta:
        mol = Chem.MolFromMolFile(entry["lig_sdf_path"], removeHs=False)
        if mol is None:
            mol = Chem.MolFromPDBFile(entry["lig_pdb_path"], removeHs=False)
        if mol is not None:
            parent_mols[entry["parent_id"]] = mol
    _n_mods_returned = 0
    _n_added = 0
    _n_dup = 0
    _n_recs = 0
    for rec in mapping_records:
        parent_id = rec["parent_id"]
        target_residue = rec["target_residue"]
        close_idxs = rec["close_atom_idxs"]
        if parent_id not in parent_mols:
            continue
        parent_mol = parent_mols[parent_id]
        parent_smiles = Chem.MolToSmiles(parent_mol)
        candidate_sites += len(close_idxs)
        if var_count_per_parent[parent_id] >= MAX_VARIANTS_PER_PARENT:
            continue
        residue_name = get_residue_name_from_label(target_residue)
        _n_recs += 1
        for atom_idx in close_idxs:
            if var_count_per_parent[parent_id] >= MAX_VARIANTS_PER_PARENT:
                break
            mods = generate_residue_specific_modifications(parent_mol, atom_idx, residue_name, max_mods=MAX_MODS_PER_SITE)
            _n_mods_returned += len(mods)
            for smi, mol, rule_category, modification_type in mods:
                if var_count_per_parent[parent_id] >= MAX_VARIANTS_PER_PARENT:
                    break
                key = MolToInchiKey(mol)
                if key in seen_inchikey:
                    duplicates_skipped += 1
                    continue
                seen_inchikey.add(key)
                vid = f"{parent_id}_var_{len(variants_rows):02d}"
                variants_rows.append({"variant_id": vid, "parent_id": parent_id, "parent_smiles": parent_smiles, "variant_smiles": smi, "modified_atom_index": atom_idx, "target_residue": target_residue, "modification_type": modification_type, "rule_category": rule_category, "num_heavy_atoms": mol.GetNumHeavyAtoms(), "mol_object": mol, "source_pdb": pdb_stem})
                log_debug({"variant_id": vid, "success": True, "mod": modification_type, "rule_category": rule_category})
                var_count_per_parent[parent_id] += 1
    _dbg("after_library", {"n_recs_processed": _n_recs, "n_mods_returned": _n_mods_returned, "n_added": _n_added, "n_dup_skipped": _n_dup, "len_variants_rows": len(variants_rows)}, "H4")

    variants_table = pd.DataFrame(variants_rows) if variants_rows else pd.DataFrame()
    variants_table.drop(columns=["mol_object"], errors="ignore").to_csv(Path(out_dir) / "variants_table.csv", index=False)
    variants_table.to_pickle(Path(out_dir) / "variants_table.pkl")
    writer = Chem.SDWriter(str(Path(out_dir) / "variants_library.sdf"))
    for _, row in variants_table.iterrows():
        mol = row.get("mol_object")
        if mol is not None:
            if mol.GetNumConformers() == 0:
                AllChem.Compute2DCoords(mol)
            writer.write(mol)
    writer.close()
    with open(Path(out_dir) / "summary.txt", "w") as f:
        f.write((chr(10)).join([f"Parents: {len(parent_mols)}", f"Candidate sites: {candidate_sites}", f"Variants: {len(variants_rows)}", f"Duplicates skipped: {duplicates_skipped}"]))
    if not variants_table.empty:
        all_tables.append(variants_table.drop(columns=["mol_object"], errors="ignore"))
    logger.info(f"Done {out_dir}: {len(variants_rows)} variants")

if all_tables:
    master_df = pd.concat(all_tables, ignore_index=True)
    master_path = Path(run_list[0][1]).parent / "fgo_all_variants.csv"
    master_df.to_csv(master_path, index=False)
    logger.info(f"Saved {master_path} ({len(master_df)} rows)")
else:
    master_df = pd.DataFrame()
    logger.info("No variant tables to merge.")

In [ ]:
# Cell 5 — Summary and examples
if not master_df.empty:
    print(master_df.head(8))
    for i, (_, row) in enumerate(master_df.head(2).iterrows()):
        print(f"\n--- Example {i+1} ---")
        print(f"Parent SMILES: {row['parent_smiles']}")
        print(f"Variant SMILES: {row['variant_smiles']}")
        print(f"modification_type: {row['modification_type']}, rule_category: {row.get('rule_category', '')}, source_pdb: {row.get('source_pdb', '')}")
    print(f"\nMaster CSV: {Path(run_list[0][1]).parent / 'fgo_all_variants.csv'}")
else:
    print("No variants generated. Check INPUT_PDB_DIR and that PDBs contain UNL ligands.")